# 07. Ensemble + Optuna Tuning
## 전략
- **CV-Public 갭(9.0 → 19.82) 원인**: test 분포 이동 + GroupKFold 내 leakage 가능성
- **개선 방향**: 
  1. Optuna로 LightGBM 하이퍼파라미터 탐색 (30 trials)
  2. CatBoost 추가 (다른 inductive bias)
  3. XGBoost 추가
  4. log1p 타깃 변환 (우편향 보정)
  5. 확장된 lag(1-6) + rolling(3,5,10) 피처
  6. 가중 앙상블

In [ ]:
import pandas as pd
import numpy as np
import sys, warnings, os
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from feature_engineering import (
    merge_layout, encode_categoricals, add_ts_features,
    add_lag_features, add_rolling_features, add_domain_features,
    get_feature_cols
)

DATA_PATH = '../data/'
TARGET    = 'avg_delay_minutes_next_30m'
SEED      = 42
N_SPLITS  = 5
print('Libraries loaded OK')

## 1. 데이터 로드 & 확장 피처 엔지니어링

In [ ]:
train_raw = pd.read_csv(DATA_PATH + 'train.csv')
test_raw  = pd.read_csv(DATA_PATH + 'test.csv')
layout    = pd.read_csv(DATA_PATH + 'layout_info.csv')
test_orig_ids = test_raw['ID'].values.copy()
print(f'Train: {train_raw.shape}, Test: {test_raw.shape}')

In [ ]:
# ── 기존 파이프라인 적용
train, test = merge_layout(train_raw.copy(), test_raw.copy(), layout)
train, test = encode_categoricals(train, test, TARGET)
train = add_ts_features(train)
test  = add_ts_features(test)

# ── 확장 Lag: 1~6
KEY_COLS_EXTENDED = [
    'low_battery_ratio', 'battery_mean', 'charge_queue_length',
    'robot_idle', 'order_inflow_15m', 'congestion_score',
    'max_zone_density', 'avg_trip_distance',
    # 추가 중요 피처
    'robot_utilization', 'task_reassign_15m', 'blocked_path_15m',
    'urgent_order_ratio', 'fault_count_15m', 'avg_recovery_time',
]

train, test = add_lag_features(train, test, key_cols=KEY_COLS_EXTENDED, lags=[1, 2, 3, 4, 5, 6])
train, test = add_rolling_features(train, test, key_cols=KEY_COLS_EXTENDED, windows=[3, 5, 10])
train = add_domain_features(train)
test  = add_domain_features(test)

assert (test['ID'].values == test_orig_ids).all(), '❌ ID 순서 오류!'
print(f'✅ 확장 피처 적용 완료')

In [ ]:
# ── 피처 컬럼 & 타깃 준비
feat_cols = get_feature_cols(train, TARGET)
print(f'피처 수: {len(feat_cols)}')

X      = train[feat_cols].values.astype(np.float32)
y      = train[TARGET].values.astype(np.float32)
y_log  = np.log1p(y)   # log1p 변환
X_test = test[feat_cols].values.astype(np.float32)
groups = train['scenario_id'].values

print(f'X: {X.shape}, X_test: {X_test.shape}')
print(f'y: mean={y.mean():.2f}, median={np.median(y):.2f}, std={y.std():.2f}')
print(f'y_log: mean={y_log.mean():.3f}, skew={pd.Series(y_log).skew():.3f}')

## 2. Optuna LightGBM 하이퍼파라미터 탐색

In [ ]:
def lgbm_objective(trial):
    params = {
        'objective'        : 'regression_l1',
        'metric'           : 'mae',
        'verbosity'        : -1,
        'boosting_type'    : 'gbdt',
        'n_estimators'     : 2000,
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 255),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq'     : 1,
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state'     : SEED,
    }
    gkf = GroupKFold(n_splits=3)  # 빠른 탐색용 3-fold
    oof = np.zeros(len(X))
    for tr_idx, val_idx in gkf.split(X, y_log, groups=groups):
        m = lgb.LGBMRegressor(**params)
        m.fit(X[tr_idx], y_log[tr_idx],
              eval_set=[(X[val_idx], y_log[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
        pred_log = m.predict(X[val_idx])
        oof[val_idx] = np.expm1(pred_log).clip(0)
    return mean_absolute_error(y, oof)

print('Optuna 탐색 시작 (30 trials, ~5-7분 예상)...')
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(lgbm_objective, n_trials=30, show_progress_bar=True)

best_params = study.best_params
best_params.update({
    'objective': 'regression_l1', 'metric': 'mae',
    'verbosity': -1, 'boosting_type': 'gbdt',
    'n_estimators': 3000, 'bagging_freq': 1,
    'random_state': SEED,
})
print(f'\n✅ Best MAE: {study.best_value:.4f}')
print(f'Best params: {best_params}')

## 3. LightGBM Full CV (최적 파라미터, log1p)

In [ ]:
gkf = GroupKFold(n_splits=N_SPLITS)

oof_lgbm  = np.zeros(len(X))
test_lgbm = np.zeros(len(X_test))
lgbm_maes = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y_log, groups=groups)):
    model = lgb.LGBMRegressor(**best_params)
    model.fit(
        X[tr_idx], y_log[tr_idx],
        eval_set=[(X[val_idx], y_log[val_idx])],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(-1)]
    )
    pred_log = model.predict(X[val_idx])
    oof_lgbm[val_idx] = np.expm1(pred_log).clip(0)
    test_lgbm += np.expm1(model.predict(X_test)).clip(0) / N_SPLITS
    
    fold_mae = mean_absolute_error(y[val_idx], oof_lgbm[val_idx])
    lgbm_maes.append(fold_mae)
    print(f'  Fold {fold+1}: MAE = {fold_mae:.4f}')

lgbm_oof_mae = mean_absolute_error(y, oof_lgbm)
print(f'\n✅ LightGBM OOF MAE: {lgbm_oof_mae:.4f} (std={np.std(lgbm_maes):.4f})')

## 4. CatBoost CV

In [ ]:
cb_params = dict(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=SEED,
    early_stopping_rounds=100,
    verbose=0,
    allow_writing_files=False,
)

oof_cb  = np.zeros(len(X))
test_cb = np.zeros(len(X_test))
cb_maes = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y_log, groups=groups)):
    model = cb.CatBoostRegressor(**cb_params)
    model.fit(
        X[tr_idx], y_log[tr_idx],
        eval_set=(X[val_idx], y_log[val_idx]),
    )
    pred_log = model.predict(X[val_idx])
    oof_cb[val_idx] = np.expm1(pred_log).clip(0)
    test_cb += np.expm1(model.predict(X_test)).clip(0) / N_SPLITS
    
    fold_mae = mean_absolute_error(y[val_idx], oof_cb[val_idx])
    cb_maes.append(fold_mae)
    print(f'  Fold {fold+1}: MAE = {fold_mae:.4f}')

cb_oof_mae = mean_absolute_error(y, oof_cb)
print(f'\n✅ CatBoost OOF MAE: {cb_oof_mae:.4f} (std={np.std(cb_maes):.4f})')

## 5. XGBoost CV

In [ ]:
xgb_params = dict(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:absoluteerror',
    eval_metric='mae',
    random_state=SEED,
    tree_method='hist',
    early_stopping_rounds=100,
    verbosity=0,
)

oof_xgb  = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))
xgb_maes = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y_log, groups=groups)):
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X[tr_idx], y_log[tr_idx],
        eval_set=[(X[val_idx], y_log[val_idx])],
        verbose=False
    )
    pred_log = model.predict(X[val_idx])
    oof_xgb[val_idx] = np.expm1(pred_log).clip(0)
    test_xgb += np.expm1(model.predict(X_test)).clip(0) / N_SPLITS
    
    fold_mae = mean_absolute_error(y[val_idx], oof_xgb[val_idx])
    xgb_maes.append(fold_mae)
    print(f'  Fold {fold+1}: MAE = {fold_mae:.4f}')

xgb_oof_mae = mean_absolute_error(y, oof_xgb)
print(f'\n✅ XGBoost OOF MAE: {xgb_oof_mae:.4f} (std={np.std(xgb_maes):.4f})')

## 6. 최적 앙상블 가중치 탐색 (OOF 기반)

In [ ]:
from scipy.optimize import minimize

oofs = np.stack([oof_lgbm, oof_cb, oof_xgb], axis=1)
names = ['LightGBM', 'CatBoost', 'XGBoost']

def neg_mae(w):
    w = np.array(w)
    w = np.abs(w) / np.abs(w).sum()  # 정규화
    blend = (oofs * w).sum(axis=1)
    return mean_absolute_error(y, blend)

# 초기값: 균등 가중치
w0 = [1/3, 1/3, 1/3]
res = minimize(neg_mae, w0, method='Nelder-Mead',
               options={'maxiter': 1000, 'xatol': 1e-6})

raw_w = np.abs(res.x)
opt_w = raw_w / raw_w.sum()

print('=== 앙상블 결과 ===')
for name, mae_val in zip(names, [lgbm_oof_mae, cb_oof_mae, xgb_oof_mae]):
    print(f'  {name:10s}: OOF MAE = {mae_val:.4f}')
print(f'  균등 앙상블: OOF MAE = {neg_mae([1/3,1/3,1/3]):.4f}')
print(f'  최적 앙상블: OOF MAE = {res.fun:.4f}')
print(f'  최적 가중치: {dict(zip(names, opt_w.round(3)))}')

## 7. 제출 파일 생성

In [ ]:
# 최적 가중 앙상블
test_ensemble = (test_lgbm * opt_w[0] +
                 test_cb   * opt_w[1] +
                 test_xgb  * opt_w[2]).clip(0)

# ID 검증
assert (test['ID'].values == test_orig_ids).all(), '❌ ID 순서 오류!'

submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')
submission['avg_delay_minutes_next_30m'] = test_ensemble

out_path = '../submissions/ensemble_lgbm_cb_xgb_optuna.csv'
submission.to_csv(out_path, index=False)

print('=== 최종 결과 ===')
print(f'  LightGBM  OOF MAE : {lgbm_oof_mae:.4f} (가중치 {opt_w[0]:.3f})')
print(f'  CatBoost  OOF MAE : {cb_oof_mae:.4f} (가중치 {opt_w[1]:.3f})')
print(f'  XGBoost   OOF MAE : {xgb_oof_mae:.4f} (가중치 {opt_w[2]:.3f})')
print(f'  앙상블    OOF MAE : {res.fun:.4f}')
print(f'  이전 Public Score : 19.8209')
print(f'  예측 통계 - mean: {test_ensemble.mean():.2f}, std: {test_ensemble.std():.2f}, max: {test_ensemble.max():.2f}')
print(f'\n제출 파일 저장: {out_path}')

## 8. 단일 최고 모델도 별도 저장 (비교용)

In [ ]:
# 단일 모델 중 최고 저장
best_single_name, best_single_pred, best_single_mae = min(
    [('lgbm', test_lgbm, lgbm_oof_mae),
     ('catboost', test_cb, cb_oof_mae),
     ('xgboost', test_xgb, xgb_oof_mae)],
    key=lambda x: x[2]
)
sub2 = pd.read_csv(DATA_PATH + 'sample_submission.csv')
sub2['avg_delay_minutes_next_30m'] = best_single_pred.clip(0)
out2 = f'../submissions/best_single_{best_single_name}_optuna.csv'
sub2.to_csv(out2, index=False)
print(f'단일 최고 모델: {best_single_name} (OOF MAE={best_single_mae:.4f}) → {out2}')